<a href="https://colab.research.google.com/github/andrebemantoro/Final-Project/blob/master/Agentic_Clustering_RBS_MultiAgent_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic Framework — Clustering + Rule-Based System + Multi-Agent (FINAL)

**Notebook ini mengikuti PROMPT FINAL (FULL) — VERSI FINAL RESMI**

Pipeline: Setup → Load Dataset → Preprocessing → Dual Embedding (MPNet & MiniLM-L12) → Clustering + Tuning → Sarwosri Labeling → Pair Generation → spaCy Parsing → Tuple Extraction → WordNet Synonym/Antonym → RBS (R1–R6) + ConfRBS → Threshold τ=0.7 → Multi-Agent + Weighted Consensus → Explainability → Export → Evaluation.

**Variabel utama:** `df_req`, `embeddings_mpnet`, `embeddings_minilm`, `df_pairs`, `df_results`.


## 1) Setup & Import Library
**Tujuan:** Menyiapkan environment Colab (install + load) dan helper utilities.

**Input:** -

**Output:** Library siap, seed global, `log()` dan `save_df()`.

**Sanity Check:** Print device & versi library inti.


In [1]:
# =========================
# 1) Setup & Import Library
# =========================
%%capture
!pip -q install pandas numpy scikit-learn sentence-transformers torch spacy nltk openpyxl tqdm
!python -m spacy download en_core_web_sm -q

import os, re, json, time, math, random
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.corpus import wordnet as wn

import spacy
nlp = spacy.load("en_core_web_sm")

import torch
from sentence_transformers import SentenceTransformer

from sklearn.cluster import KMeans, AgglomerativeClustering, MeanShift, estimate_bandwidth
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def log(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

def save_df(df: pd.DataFrame, filename: str):
    os.makedirs("outputs", exist_ok=True)
    path = os.path.join("outputs", filename)
    if filename.lower().endswith(".xlsx"):
        df.to_excel(path, index=False)
    else:
        df.to_csv(path, index=False)
    log(f"Saved: {path} ({len(df):,} rows)")

log(f"Device: {DEVICE}")
log(f"pandas={pd.__version__}, numpy={np.__version__}, torch={torch.__version__}")


## 2) Load Dataset
**Tujuan:** Mount Google Drive dan load OPENCOSS.xlsx ke `df_req`.

**Input:** Path dataset `/content/drive/MyDrive/Datasets ML/_Dataset/OPENCOSS.xlsx`.

**Output:** `df_req` dengan kolom `id | requirements | class | conflict_id`.

**Sanity Check:** Tampilkan distribusi class dan 5 baris contoh.


In [2]:
# ==============
# 2) Load Dataset
# ==============
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/Datasets ML/_Dataset/OPENCOSS.xlsx"
#DATA_PATH = "/content/drive/MyDrive/Datasets ML/_Dataset/ecommerce.xlsx"
assert os.path.exists(DATA_PATH), f"Dataset not found: {DATA_PATH}"

df_req = pd.read_excel(DATA_PATH)
df_req.columns = [str(c).strip().lower() for c in df_req.columns]

# Map common variants -> expected schema
col_map = {}
if "requirements" not in df_req.columns:
    for cand in ["requirement", "requirement_text", "text", "sentence"]:
        if cand in df_req.columns:
            col_map[cand] = "requirements"
            break
if "id" not in df_req.columns:
    for cand in ["req_id", "requirement_id", "idx"]:
        if cand in df_req.columns:
            col_map[cand] = "id"
            break
if col_map:
    df_req = df_req.rename(columns=col_map)

required = {"id","requirements","class"}
missing = required - set(df_req.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found: {df_req.columns.tolist()}")

if "conflict_id" not in df_req.columns:
    df_req["conflict_id"] = np.nan

df_req = df_req[["id","requirements","class","conflict_id"]].copy()
df_req["requirements"] = df_req["requirements"].astype(str)

# Basic cleanup
df_req = df_req.dropna(subset=["requirements"])
df_req = df_req[df_req["requirements"].str.strip().ne("")]
df_req = df_req.drop_duplicates(subset=["id"]).reset_index(drop=True)

log(f"Loaded df_req: {df_req.shape}")
print(df_req["class"].astype(str).str.lower().value_counts(dropna=False))
display(df_req.head())


Mounted at /content/drive
[16:37:25] Loaded df_req: (117, 4)
class
neutral     97
conflict    20
Name: count, dtype: int64


,id,requirements,class,conflict_id
0,1,The OPENCOSS platform shall show the set of ev...,Neutral,NaN
1,2,The OPENCOSS platform shall provide users with...,Neutral,NaN
2,3,The OPENCOSS platform must provide users with ...,Conflict,4.0
3,4,The OPENCOSS platform shall provide users with...,Conflict,3.0
4,5,The OPENCOSS platform shall provide users with...,Neutral,NaN


## 3) Preprocessing
**Tujuan:** Membersihkan teks untuk embedding & parsing.

**Input:** `df_req.requirements`

**Proses:** case folding, remove punctuation, normalize whitespace, optional lemmatization (spaCy).

**Output:** Kolom `text_raw`, `text_clean`, `text_lemma`.

**Sanity Check:** Tampilkan 3 contoh sebelum/sesudah.


In [3]:
# ==============
# 3) Preprocessing
# ==============
PUNCT_RE = re.compile(r"[^a-zA-Z0-9\s]")

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

def preprocess_text(text: str) -> Tuple[str,str,str]:
    text_raw = str(text)
    text_clean = normalize_ws(PUNCT_RE.sub(" ", text_raw.lower()))
    # lemmatization (optional but enabled by default)
    doc = nlp(text_clean)
    text_lemma = normalize_ws(" ".join([t.lemma_ for t in doc if not t.is_space]))
    return text_raw, text_clean, text_lemma

tqdm.pandas()
df_req[["text_raw","text_clean","text_lemma"]] = df_req["requirements"].progress_apply(
    lambda x: pd.Series(preprocess_text(x))
)

log("Preprocessing done.")
display(df_req[["requirements","text_clean","text_lemma"]].head(3))


100%|██████████| 117/117 [00:03<00:00, 37.47it/s]

[16:37:28] Preprocessing done.


,requirements,text_clean,text_lemma
0,The OPENCOSS platform shall show the set of ev...,the opencoss platform shall show the set of ev...,the opencoss platform shall show the set of ev...
1,The OPENCOSS platform shall provide users with...,the opencoss platform shall provide users with...,the opencoss platform shall provide user with ...
2,The OPENCOSS platform must provide users with ...,the opencoss platform must provide users with ...,the opencoss platform must provide user with t...


## 4) Embedding (DUAL SBERT MODELS)
**Tujuan:** Membuat 2 embedding berbeda untuk clustering.

**Input:** `df_req.text_lemma`

**Proses:**
- Model 1: `all-mpnet-base-v2` → untuk KMeans
- Model 2: `all-MiniLM-L12-v2` → untuk Agglomerative & MeanShift

**Output:** `embeddings_mpnet`, `embeddings_minilm`

**Sanity Check:** Log shape & cosine similarity contoh.


In [4]:
# =============================
# 4) Embedding (Dual SBERT)
# =============================
MODEL_MPNET = "all-mpnet-base-v2"
MODEL_MINILM = "all-MiniLM-L12-v2"

def embed_texts(model_name: str, texts: List[str], batch_size: int = 64) -> np.ndarray:
    t0 = time.time()
    model = SentenceTransformer(model_name, device=DEVICE)
    embs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    log(f"Embedded {model_name} -> shape={embs.shape}, time={time.time()-t0:.1f}s")
    return embs

texts = df_req["text_lemma"].astype(str).tolist()

embeddings_mpnet = embed_texts(MODEL_MPNET, texts, batch_size=64)
embeddings_minilm = embed_texts(MODEL_MINILM, texts, batch_size=64)

# sanity: cosine sim on first 3 (MPNet)
if len(texts) >= 3:
    sim = cosine_similarity(embeddings_mpnet[:3], embeddings_mpnet[:3])
    log("Cosine similarity sample (MPNet, 3x3):")
    print(np.round(sim, 3))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

[16:38:29] Embedded all-MiniLM-L12-v2 -> shape=(117, 384), time=10.7s
[16:38:29] Cosine similarity sample (MPNet, 3x3):
[[1.    0.918 0.863]
 [0.918 1.    0.943]
 [0.863 0.943 1.   ]]


## 5) Clustering + Hyperparameter Tuning + Sarwosri Labeling
**Tujuan:** Tuning internal metrics lalu pilih konfigurasi terbaik per metode.

**Input:**
- KMeans ← `embeddings_mpnet`
- Agglomerative ← `embeddings_minilm`
- MeanShift ← `embeddings_minilm`

**Proses:** Grid search + Silhouette, Calinski-Harabasz, Davies-Bouldin.

**Output:** Label cluster per metode, tabel tuning, `df_cluster_summary`.

**Sanity Check:** Tampilkan best config & ringkasan cluster.


In [5]:
import math

def internal_metrics(X: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
    n_unique = len(set(labels))
    out = {"n_clusters": int(n_unique)}
    if n_unique <= 1 or n_unique >= len(labels):
        out["silhouette"] = float("nan")
        out["calinski_harabasz"] = float("nan")
        out["davies_bouldin"] = float("nan")
        return out
    out["silhouette"] = float(silhouette_score(X, labels))
    out["calinski_harabasz"] = float(calinski_harabasz_score(X, labels))
    out["davies_bouldin"] = float(davies_bouldin_score(X, labels))
    return out

def pick_best(df: pd.DataFrame) -> pd.Series:
    d = df.copy()
    d["sil_rank"] = d["silhouette"].rank(ascending=False, na_option="bottom")
    d["ch_rank"] = d["calinski_harabasz"].rank(ascending=False, na_option="bottom")
    d["db_rank"] = d["davies_bouldin"].rank(ascending=True, na_option="bottom")
    d["rank_sum"] = d["sil_rank"] + d["ch_rank"] + d["db_rank"]
    return d.sort_values("rank_sum").iloc[0]

def tune_kmeans(X: np.ndarray, k_list: List[int], init_list: List[str], max_iter_list: List[int]):
    rows = []
    for k in k_list:
        for init in init_list:
            for mi in max_iter_list:
                km = KMeans(n_clusters=k, init=init, max_iter=mi, random_state=SEED, n_init="auto")
                labels = km.fit_predict(X)
                m = internal_metrics(X, labels)
                rows.append({"method":"kmeans","embedding_type":"mpnet","k":k,"init":init,"max_iter":mi, **m})
    res = pd.DataFrame(rows)
    best = pick_best(res)
    best_cfg = {"n_clusters": int(best["k"]), "init": str(best["init"]), "max_iter": int(best["max_iter"])}
    best_labels = KMeans(**best_cfg, random_state=SEED, n_init="auto").fit_predict(X)
    return best_labels, res, best_cfg

def tune_agglomerative(X: np.ndarray, k_list: List[int], linkages: List[str]):
    rows = []
    for k in k_list:
        for link in linkages:
            agg = AgglomerativeClustering(n_clusters=k, linkage=link)
            labels = agg.fit_predict(X)
            m = internal_metrics(X, labels)
            rows.append({"method":"agglomerative","embedding_type":"minilm","k":k,"linkage":link, **m})
    res = pd.DataFrame(rows)
    best = pick_best(res)
    best_cfg = {"n_clusters": int(best["k"]), "linkage": str(best["linkage"])}
    best_labels = AgglomerativeClustering(**best_cfg).fit_predict(X)
    return best_labels, res, best_cfg

def tune_meanshift(X: np.ndarray, bw_list: List[float]):
    rows = []
    for bw in bw_list:
        ms = MeanShift(bandwidth=bw, bin_seeding=True)
        labels = ms.fit_predict(X)
        m = internal_metrics(X, labels)
        rows.append({"method":"meanshift","embedding_type":"minilm","bandwidth":float(bw), **m})
    res = pd.DataFrame(rows)
    best = pick_best(res)
    best_cfg = {"bandwidth": float(best["bandwidth"])}
    best_labels = MeanShift(**best_cfg, bin_seeding=True).fit_predict(X)
    return best_labels, res, best_cfg

# --- grids (adjust if needed) ---
K_LIST = [5, 8, 10, 12, 15]
INIT_LIST = ["k-means++", "random"]
MAX_ITER_LIST = [200, 300]
LINKAGES = ["ward", "average", "complete"]

bw_est = estimate_bandwidth(embeddings_minilm, quantile=0.2, n_samples=min(2000, len(df_req)))
if not (bw_est > 0) or math.isnan(bw_est):
    bw_est = 0.8 # Fallback if estimation fails

# Ensure the estimated bandwidth itself is not too low
# Set a minimum effective bandwidth to avoid the ValueError
MIN_EFFECTIVE_BANDWIDTH = 1.0
if bw_est < MIN_EFFECTIVE_BANDWIDTH:
    log(f"Adjusting estimated bandwidth from {bw_est:.2f} to {MIN_EFFECTIVE_BANDWIDTH:.2f} to avoid MeanShift errors.")
    bw_est = MIN_EFFECTIVE_BANDWIDTH

# Generate a list of bandwidth candidates around the (potentially adjusted) bw_est
# Ensure all candidates are above MIN_EFFECTIVE_BANDWIDTH
BW_CANDIDATES = [
    bw_est * 0.5,
    bw_est,
    bw_est * 1.5,
    bw_est * 2.0
]
BW_LIST = sorted(list(set([max(MIN_EFFECTIVE_BANDWIDTH, bw) for bw in BW_CANDIDATES])))

log("Tuning KMeans (MPNet)...")
labels_km, tune_km, cfg_km = tune_kmeans(embeddings_mpnet, K_LIST, INIT_LIST, MAX_ITER_LIST)
log(f"Best KMeans cfg: {cfg_km}")

log("Tuning Agglomerative (MiniLM)...")
labels_ag, tune_ag, cfg_ag = tune_agglomerative(embeddings_minilm, K_LIST, LINKAGES)
log(f"Best Agglomerative cfg: {cfg_ag}")

log("Tuning MeanShift (MiniLM)...")
labels_ms, tune_ms, cfg_ms = tune_meanshift(embeddings_minilm, BW_LIST)
log(f"Best MeanShift cfg: {cfg_ms}")

df_tuning = pd.concat([tune_km, tune_ag, tune_ms], ignore_index=True)
save_df(df_tuning, "clustering_tuning_metrics.csv")
display(df_tuning.sort_values(["method","silhouette"], ascending=[True, False]).head(10))

# attach cluster labels
df_req["cluster_kmeans"] = labels_km
df_req["cluster_agglomerative"] = labels_ag
df_req["cluster_meanshift"] = labels_ms

[16:38:29] Adjusting estimated bandwidth from 0.61 to 1.00 to avoid MeanShift errors.
[16:38:29] Tuning KMeans (MPNet)...
[16:38:30] Best KMeans cfg: {'n_clusters': 12, 'init': 'k-means++', 'max_iter': 200}
[16:38:30] Tuning Agglomerative (MiniLM)...
[16:38:30] Best Agglomerative cfg: {'n_clusters': 15, 'linkage': 'ward'}
[16:38:30] Tuning MeanShift (MiniLM)...
[16:38:30] Best MeanShift cfg: {'bandwidth': 1.0}
[16:38:30] Saved: outputs/clustering_tuning_metrics.csv (38 rows)


,method,embedding_type,k,init,max_iter,n_clusters,silhouette,calinski_harabasz,davies_bouldin,linkage,bandwidth
34,agglomerative,minilm,15.0,NaN,NaN,15,0.190210,9.542368,1.471081,complete,NaN
32,agglomerative,minilm,15.0,NaN,NaN,15,0.188612,10.033018,1.466540,ward,NaN
33,agglomerative,minilm,15.0,NaN,NaN,15,0.185138,8.428481,1.363224,average,NaN
31,agglomerative,minilm,12.0,NaN,NaN,12,0.172766,9.507195,1.603867,complete,NaN
30,agglomerative,minilm,12.0,NaN,NaN,12,0.172124,8.872096,1.516790,average,NaN
29,agglomerative,minilm,12.0,NaN,NaN,12,0.165367,10.497234,1.591030,ward,NaN
26,agglomerative,minilm,10.0,NaN,NaN,10,0.164679,11.116575,1.768952,ward,NaN
23,agglomerative,minilm,8.0,NaN,NaN,8,0.158544,11.201858,1.910787,ward,NaN
28,agglomerative,minilm,10.0,NaN,NaN,10,0.156961,10.196894,1.858773,complete,NaN
27,agglomerative,minilm,10.0,NaN,NaN,10,0.153885,7.968682,1.557511,average,NaN


In [6]:
# =====================================
# 5C) Sarwosri Cluster Conflict Labeling
# =====================================
def sarwosri_cluster_summary(df: pd.DataFrame, cluster_col: str, method: str, embedding_type: str) -> pd.DataFrame:
    rows = []
    for cid, sub in df.groupby(cluster_col):
        size = len(sub)
        n_conf = int((sub["class"].astype(str).str.lower() == "conflict").sum())
        n_neu = int((sub["class"].astype(str).str.lower() == "neutral").sum())
        if size == 1:
            clabel = "neutral"
        elif n_conf >= n_neu:
            clabel = "conflict"
        else:
            clabel = "neutral"
        rows.append({
            "cluster_id": cid,
            "cluster_size": size,
            "jumlah_conflict": n_conf,
            "jumlah_neutral": n_neu,
            "cluster_label": clabel,
            "method": method,
            "embedding_type": embedding_type,
            "cluster_col": cluster_col
        })
    return pd.DataFrame(rows)

sum_km = sarwosri_cluster_summary(df_req, "cluster_kmeans", "kmeans", "mpnet")
sum_ag = sarwosri_cluster_summary(df_req, "cluster_agglomerative", "agglomerative", "minilm")
sum_ms = sarwosri_cluster_summary(df_req, "cluster_meanshift", "meanshift", "minilm")

df_cluster_summary = pd.concat([sum_km, sum_ag, sum_ms], ignore_index=True)
save_df(df_cluster_summary, "cluster_summary_sarwosri.csv")
display(df_cluster_summary.head())


[16:38:30] Saved: outputs/cluster_summary_sarwosri.csv (28 rows)


,cluster_id,cluster_size,jumlah_conflict,jumlah_neutral,cluster_label,method,embedding_type,cluster_col
0,0,3,0,3,neutral,kmeans,mpnet,cluster_kmeans
1,1,12,4,8,neutral,kmeans,mpnet,cluster_kmeans
2,2,16,2,14,neutral,kmeans,mpnet,cluster_kmeans
3,3,8,2,6,neutral,kmeans,mpnet,cluster_kmeans
4,4,19,0,19,neutral,kmeans,mpnet,cluster_kmeans


In [7]:
# =====================
# 5D) Scenario Mode
# =====================
scenario_mode = 1  # set to 2 to merge singletons into one cluster
log(f"scenario_mode={scenario_mode}")

def apply_scenario(df: pd.DataFrame, cluster_col: str, mode: int) -> pd.Series:
    counts = df[cluster_col].value_counts()
    is_single = df[cluster_col].map(counts) == 1
    if mode == 2:
        new = df[cluster_col].copy()
        new[is_single] = -1
        return new
    return df[cluster_col]

df_req["cluster_kmeans_scn"] = apply_scenario(df_req, "cluster_kmeans", scenario_mode)
df_req["cluster_agglomerative_scn"] = apply_scenario(df_req, "cluster_agglomerative", scenario_mode)
df_req["cluster_meanshift_scn"] = apply_scenario(df_req, "cluster_meanshift", scenario_mode)


[16:38:30] scenario_mode=1


In [8]:
# =========================
# 5E) Cluster-Level Evaluation
# =========================
def eval_cluster_level(df: pd.DataFrame, label_summary: pd.DataFrame, cluster_col: str) -> Dict[str,float]:
    # map cluster->cluster_label then compare to row class (proxy cluster-level)
    m = dict(zip(label_summary["cluster_id"], label_summary["cluster_label"]))
    y_pred = df[cluster_col].map(m).fillna("neutral").astype(str).str.lower().map({"conflict":1,"neutral":0}).astype(int)
    y_true = df["class"].astype(str).str.lower().map({"conflict":1,"neutral":0}).fillna(0).astype(int)
    acc = accuracy_score(y_true, y_pred)
    p,r,f1,_ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {"cluster_col": cluster_col, "accuracy": float(acc), "precision": float(p), "recall": float(r), "f1": float(f1)}

ev_rows = [
    eval_cluster_level(df_req, sum_km, "cluster_kmeans"),
    eval_cluster_level(df_req, sum_ag, "cluster_agglomerative"),
    eval_cluster_level(df_req, sum_ms, "cluster_meanshift"),
]
df_cluster_eval = pd.DataFrame(ev_rows)
save_df(df_cluster_eval, "cluster_level_eval.csv")
display(df_cluster_eval)


[16:38:31] Saved: outputs/cluster_level_eval.csv (3 rows)


,cluster_col,accuracy,precision,recall,f1
0,cluster_kmeans,0.829060,0.000000,0.0,0.000000
1,cluster_agglomerative,0.837607,0.666667,0.1,0.173913
2,cluster_meanshift,0.829060,0.000000,0.0,0.000000


## 6) Pair Generation
**Tujuan:** Membentuk pasangan i<j hanya dalam cluster yang sama.

**Input:** `df_req` + kolom cluster scenario.

**Output:** `df_pairs` dengan kolom sesuai prompt.

**Sanity Check:** Tampilkan jumlah pair dan head.


In [9]:
# ==============================
# 6) Pair Generation Intra-Cluster
# ==============================
def generate_pairs(df: pd.DataFrame, cluster_col: str, method: str, embedding_type: str) -> pd.DataFrame:
    pairs = []
    for cid, sub in df.groupby(cluster_col):
        if len(sub) < 2:
            continue
        ids = sub["id"].tolist()
        texts1 = sub["text_lemma"].tolist()
        for i in range(len(ids)):
            for j in range(i+1, len(ids)):
                pairs.append({
                    "req_id_1": ids[i],
                    "req_id_2": ids[j],
                    "text_1": texts1[i],
                    "text_2": texts1[j],
                    "cluster_id": cid,
                    "method": method,
                    "embedding_type": embedding_type,
                })
    return pd.DataFrame(pairs)

df_pairs = pd.concat([
    generate_pairs(df_req, "cluster_kmeans_scn", "kmeans", "mpnet"),
    generate_pairs(df_req, "cluster_agglomerative_scn", "agglomerative", "minilm"),
    generate_pairs(df_req, "cluster_meanshift_scn", "meanshift", "minilm"),
], ignore_index=True)

log(f"df_pairs shape: {df_pairs.shape}")
save_df(df_pairs, "pairs_intra_cluster.csv")
display(df_pairs.head())


[16:38:31] df_pairs shape: (8035, 7)
[16:38:31] Saved: outputs/pairs_intra_cluster.csv (8,035 rows)


,req_id_1,req_id_2,text_1,text_2,cluster_id,method,embedding_type
0,44,79,the opencoss platform shall provide user with ...,the opencoss platform shall be able to generat...,0,kmeans,mpnet
1,44,80,the opencoss platform shall provide user with ...,the opencoss platform shall be able to generat...,0,kmeans,mpnet
2,79,80,the opencoss platform shall be able to generat...,the opencoss platform shall be able to generat...,0,kmeans,mpnet
3,29,30,when an evidence item of an assurance project ...,when an evidence item of an assurance project ...,1,kmeans,mpnet
4,29,31,when an evidence item of an assurance project ...,when an evidence traceability link of an assur...,1,kmeans,mpnet


## 7) POS Tagging (spaCy)
**Tujuan:** Parsing dependency untuk fitur tuple.

**Input:** `df_pairs.text_1`, `df_pairs.text_2`

**Output:** `parse_1`, `parse_2`

**Sanity Check:** Tampilkan 1 contoh parse.


In [10]:
# ================================
# 7) POS Tagging & Dependency Parse
# ================================
def spacy_parse(text: str) -> Dict[str, Any]:
    doc = nlp(text)
    root = ""
    nsubj = ""
    dobj = ""
    neg = []
    aux = []
    advmod = []
    for t in doc:
        if t.dep_ == "ROOT":
            root = t.lemma_
        if t.dep_ in ("nsubj","nsubjpass") and not nsubj:
            nsubj = t.text
        if t.dep_ == "dobj" and not dobj:
            dobj = t.text
        if t.dep_ == "neg":
            neg.append(t.text)
        if t.dep_ == "aux":
            aux.append(t.text)
        if t.dep_ == "advmod":
            advmod.append(t.text)
    return {
        "root": root,
        "nsubj": nsubj,
        "dobj": dobj,
        "neg": neg,
        "aux": aux,
        "advmod": advmod
    }

tqdm.pandas()
df_pairs["parse_1"] = df_pairs["text_1"].progress_apply(spacy_parse)
df_pairs["parse_2"] = df_pairs["text_2"].progress_apply(spacy_parse)

display(df_pairs[["req_id_1","req_id_2","parse_1","parse_2"]].head(1))


100%|██████████| 8035/8035 [01:31<00:00, 87.75it/s]


,req_id_1,req_id_2,parse_1,parse_2
0,44,79,"{'root': 'provide', 'nsubj': 'platform', 'dobj...","{'root': 'be', 'nsubj': 'platform', 'dobj': 'r..."


## 8) Tuple Extraction
**Tujuan:** Menghasilkan tuple: agent, operation, input, output, restriction, event.

**Input:** parse dict + text.

**Output:** `tuple_1`, `tuple_2`

**Sanity Check:** Tampilkan 2 tuple.


In [11]:
# =================
# 8) Tuple Extraction
# =================
LEX_RESTRICTIONS = ["only","once","at most","at least","never","cannot","must","shall"]

def extract_tuple(parse: Dict[str,Any], text: str) -> Dict[str,str]:
    agent = parse.get("nsubj","")
    operation = parse.get("root","")
    inp = parse.get("dobj","")
    # baseline: output & event can be improved for your dataset patterns
    out = ""
    event = ""
    low = text.lower()
    hits = [c for c in LEX_RESTRICTIONS if c in low]
    restriction = ",".join(sorted(set(hits))) if hits else ""
    return {
        "agent": agent,
        "operation": operation,
        "input": inp,
        "output": out,
        "restriction": restriction,
        "event": event
    }

df_pairs["tuple_1"] = df_pairs.apply(lambda r: extract_tuple(r["parse_1"], r["text_1"]), axis=1)
df_pairs["tuple_2"] = df_pairs.apply(lambda r: extract_tuple(r["parse_2"], r["text_2"]), axis=1)

display(df_pairs[["req_id_1","req_id_2","tuple_1","tuple_2"]].head(2))


,req_id_1,req_id_2,tuple_1,tuple_2
0,44,79,"{'agent': 'platform', 'operation': 'provide', ...","{'agent': 'platform', 'operation': 'be', 'inpu..."
1,44,80,"{'agent': 'platform', 'operation': 'provide', ...","{'agent': 'platform', 'operation': 'be', 'inpu..."


## 9) WordNet Synonym/Antonym
**Tujuan:** Membangun fitur sinonim/antonim untuk operation/input/output/restriction dan IO.

**Output fitur:** syn_op, ant_op, syn_in, ant_in, syn_out, ant_out, syn_res, ant_res, syn_out_in, ant_out_in.

**Sanity Check:** Tampilkan beberapa kolom fitur.


In [12]:
# =========================
# 9) WordNet Synonym/Antonym
# =========================
def _norm(x: str) -> str:
    return re.sub(r"\s+", " ", str(x).strip().lower())

def _synsets(term: str):
    term = _norm(term)
    if not term:
        return []
    return wn.synsets(term) + wn.synsets(term.replace(" ", "_"))

def is_synonym(a: str, b: str) -> bool:
    a, b = _norm(a), _norm(b)
    if not a or not b:
        return False
    if a == b:
        return True
    sa, sb = _synsets(a), _synsets(b)
    if not sa or not sb:
        return False
    la = {l.name().lower() for s in sa for l in s.lemmas()}
    lb = {l.name().lower() for s in sb for l in s.lemmas()}
    return len(la & lb) > 0

def is_antonym(a: str, b: str) -> bool:
    a, b = _norm(a), _norm(b)
    if not a or not b or a == b:
        return False
    sa = _synsets(a)
    if not sa:
        return False
    ants = set()
    for s in sa:
        for l in s.lemmas():
            for ant in l.antonyms():
                ants.add(ant.name().lower())
    sb = _synsets(b)
    lb = {l.name().lower() for s in sb for l in s.lemmas()} if sb else {b.replace(" ", "_")}
    return len(ants & lb) > 0

def wn_features(t1: Dict[str,str], t2: Dict[str,str]) -> Dict[str,int]:
    op1, op2 = t1.get("operation",""), t2.get("operation","")
    in1, in2 = t1.get("input",""), t2.get("input","")
    out1, out2 = t1.get("output",""), t2.get("output","")
    r1, r2 = t1.get("restriction",""), t2.get("restriction","")

    syn_op, ant_op = int(is_synonym(op1,op2)), int(is_antonym(op1,op2))
    syn_in, ant_in = int(is_synonym(in1,in2)), int(is_antonym(in1,in2))
    syn_out, ant_out = int(is_synonym(out1,out2)), int(is_antonym(out1,out2))
    syn_res, ant_res = int(is_synonym(r1,r2)), int(is_antonym(r1,r2))

    # IO (both directions)
    syn_out_in = int(is_synonym(out1,in2) or is_synonym(out2,in1))
    ant_out_in = int(is_antonym(out1,in2) or is_antonym(out2,in1))

    return {
        "syn_op": syn_op, "ant_op": ant_op,
        "syn_in": syn_in, "ant_in": ant_in,
        "syn_out": syn_out, "ant_out": ant_out,
        "syn_res": syn_res, "ant_res": ant_res,
        "syn_out_in": syn_out_in, "ant_out_in": ant_out_in
    }

feat = df_pairs.apply(lambda r: pd.Series(wn_features(r["tuple_1"], r["tuple_2"])), axis=1)
df_pairs = pd.concat([df_pairs, feat], axis=1)
display(df_pairs[["req_id_1","req_id_2","syn_op","ant_op","syn_in","ant_in","syn_res","ant_res","syn_out_in","ant_out_in"]].head())


,req_id_1,req_id_2,syn_op,ant_op,syn_in,ant_in,syn_res,ant_res,syn_out_in,ant_out_in
0,44,79,0,0,0,0,1,0,0,0
1,44,80,0,0,0,0,1,0,0,0
2,79,80,1,0,1,0,1,0,0,0
3,29,30,1,0,1,0,1,0,0,0
4,29,31,1,0,1,0,1,0,0,0


## 10) Rule-Based System (R1–R6) — Identik dengan Tabel
**Tujuan:** Deteksi konflik berbasis pola P1–P4 dan trigger synonym/antonym.

**Output per pair:** active_rules, rbs_label, rbs_conf (dihitung tahap 15), rbs_explain.


In [13]:
# =====================================
# 10) RBS (R1–R6) — Identik dengan Tabel
# =====================================
def p1_same_agent(t1: Dict[str,str], t2: Dict[str,str]) -> bool:
    return _norm(t1.get("agent","")) != "" and _norm(t1.get("agent","")) == _norm(t2.get("agent",""))

def p2_same_operation(t1: Dict[str,str], t2: Dict[str,str]) -> bool:
    op1, op2 = t1.get("operation",""), t2.get("operation","")
    return is_synonym(op1, op2) and not is_antonym(op1, op2)

def p3_same_event(t1: Dict[str,str], t2: Dict[str,str]) -> bool:
    e1, e2 = _norm(t1.get("event","")), _norm(t2.get("event",""))
    # soft: both empty -> treat as same context
    if not e1 and not e2:
        return True
    return e1 == e2

def p4_same_scope(text1: str, text2: str) -> bool:
    # soft scope: token Jaccard
    s1 = set(_norm(text1).split())
    s2 = set(_norm(text2).split())
    if not s1 or not s2:
        return False
    j = len(s1 & s2) / max(1, len(s1 | s2))
    return j >= 0.25

def restriction_asym(t1: Dict[str,str], t2: Dict[str,str]) -> bool:
    r1, r2 = _norm(t1.get("restriction","")), _norm(t2.get("restriction",""))
    return (bool(r1) and not bool(r2)) or (bool(r2) and not bool(r1))

def rbs_apply(row: pd.Series) -> Dict[str,Any]:
    t1, t2 = row["tuple_1"], row["tuple_2"]
    P1 = p1_same_agent(t1, t2)
    P2 = p2_same_operation(t1, t2)
    P3 = p3_same_event(t1, t2)
    P4 = p4_same_scope(row["text_1"], row["text_2"])

    active_rules = []
    explain = []

    # R1 — Conflict on Input | P1,P2,P3,P4 | Syn/Ant(Input)
    if P1 and P2 and P3 and P4 and (row["syn_in"] == 1 or row["ant_in"] == 1):
        active_rules.append("R1")
        explain.append("R1 triggered: P1,P2,P3,P4 and synonym/antonym on Input.")

    # R2 — Conflict on Output | P1,P2,P3,P4 | Syn/Ant(Output)
    if P1 and P2 and P3 and P4 and (row["syn_out"] == 1 or row["ant_out"] == 1):
        active_rules.append("R2")
        explain.append("R2 triggered: P1,P2,P3,P4 and synonym/antonym on Output.")

    # R3 — Conflict on Restriction | P1,P2,P3,P4 | Syn/Ant(Restriction) (+ asymmetry)
    if P1 and P2 and P3 and P4 and ((row["syn_res"] == 1 or row["ant_res"] == 1) or restriction_asym(t1, t2)):
        active_rules.append("R3")
        explain.append("R3 triggered: P1,P2,P3,P4 and restriction conflict/asymmetry.")

    # R4 — Input-Output Conflict | P1,P2,P3 | Syn/Ant(Input–Output)
    if P1 and P2 and P3 and (row["syn_out_in"] == 1 or row["ant_out_in"] == 1):
        active_rules.append("R4")
        explain.append("R4 triggered: P1,P2,P3 and synonym/antonym on Input–Output mapping.")

    # R5 — Output-Output Conflict | P1,P2,P3 | Syn/Ant(Output–Output)
    if P1 and P2 and P3 and (row["syn_out"] == 1 or row["ant_out"] == 1):
        active_rules.append("R5")
        explain.append("R5 triggered: P1,P2,P3 and synonym/antonym on Output–Output.")

    # R6 — Conflict on Operation | P1 | Syn/Ant(Operation)
    if P1 and (row["syn_op"] == 1 or row["ant_op"] == 1):
        active_rules.append("R6")
        explain.append("R6 triggered: P1 and synonym/antonym on Operation.")

    rbs_label = "Conflict" if active_rules else "Non-Conflict"
    return {
        "active_rules": active_rules,
        "rbs_label": rbs_label,
        "rbs_explain": " ".join(explain),
        # rbs_conf computed later (stage 15)
    }

out = df_pairs.apply(lambda r: pd.Series(rbs_apply(r)), axis=1)
df_pairs = pd.concat([df_pairs, out], axis=1)

display(df_pairs[["req_id_1","req_id_2","active_rules","rbs_label","rbs_explain"]].head(5))


,req_id_1,req_id_2,active_rules,rbs_label,rbs_explain
0,44,79,[],Non-Conflict,
1,44,80,[],Non-Conflict,
2,79,80,"[R1, R3, R6]",Conflict,"R1 triggered: P1,P2,P3,P4 and synonym/antonym ..."
3,29,30,"[R1, R3, R6]",Conflict,"R1 triggered: P1,P2,P3,P4 and synonym/antonym ..."
4,29,31,[],Non-Conflict,


## 15) Precision-based weights + ConfRBS (probabilistik)
**Tujuan:** Hitung bobot rule (w_r) dari TP/FP pada data validasi dan hitung `Conf_RBS = 1 − ∏(1 − w_r)`.

**Catatan:** Jika kamu punya ground-truth pair-level via `conflict_id`, ganti proxy label di bawah.


In [14]:
# ==========================================
# 15A) Build pair-level ground truth (proxy)
# ==========================================
# Proxy: pair is conflict if BOTH rows are labeled conflict.
# Replace with true pair-level mapping using conflict_id when available.
id_to_class = df_req.set_index("id")["class"].astype(str).str.lower().to_dict()

def y_true_proxy(rid1, rid2) -> int:
    return int(id_to_class.get(rid1,"neutral") == "conflict" and id_to_class.get(rid2,"neutral") == "conflict")

df_pairs["y_true_pair"] = df_pairs.apply(lambda r: y_true_proxy(r["req_id_1"], r["req_id_2"]), axis=1)

df_val, df_test = train_test_split(
    df_pairs, test_size=0.7, random_state=SEED, stratify=df_pairs["y_true_pair"]
)
log(f"Validation pairs: {len(df_val):,}, Test pairs: {len(df_test):,}")

# =====================================
# 15B) Rule weights (precision-based)
# =====================================
RULES = ["R1","R2","R3","R4","R5","R6"]

def rule_precision(df: pd.DataFrame, rule: str) -> float:
    active = df["active_rules"].apply(lambda xs: rule in xs)
    tp = int(((active == True) & (df["y_true_pair"] == 1)).sum())
    fp = int(((active == True) & (df["y_true_pair"] == 0)).sum())
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

w_rule = {r: rule_precision(df_val, r) for r in RULES}
df_w_rule = pd.DataFrame([{"rule":k, "w_r":v} for k,v in w_rule.items()]).sort_values("rule")
save_df(df_w_rule, "rule_weights_precision.csv")
display(df_w_rule)

def conf_rbs(active_rules: List[str]) -> float:
    prod = 1.0
    for r in active_rules:
        prod *= (1.0 - float(w_rule.get(r, 0.0)))
    return 1.0 - prod

df_pairs["rbs_conf"] = df_pairs["active_rules"].apply(conf_rbs)
df_pairs["rbs_rule"] = df_pairs["active_rules"].apply(lambda ar: max(ar, key=lambda r: w_rule.get(r,0.0)) if ar else "")

display(df_pairs[["req_id_1","req_id_2","active_rules","rbs_rule","rbs_conf","rbs_label"]].head(5))


[16:41:45] Validation pairs: 2,410, Test pairs: 5,625
[16:41:45] Saved: outputs/rule_weights_precision.csv (6 rows)


,rule,w_r
0,R1,0.019417
1,R2,0.000000
2,R3,0.007407
3,R4,0.000000
4,R5,0.000000
5,R6,0.017065


,req_id_1,req_id_2,active_rules,rbs_rule,rbs_conf,rbs_label
0,44,79,[],,0.000000,Non-Conflict
1,44,80,[],,0.000000,Non-Conflict
2,79,80,"[R1, R3, R6]",R1,0.043291,Conflict
3,29,30,"[R1, R3, R6]",R1,0.043291,Conflict
4,29,31,[],,0.000000,Non-Conflict


## 11) Threshold τ + Escalation
**τ = 0.7**. Jika `rbs_conf >= τ` accept; jika `< τ` escalate ke multi-agent.


In [15]:
# =============================
# 11) Threshold τ + Escalation
# =============================
TAU = 0.7
df_pairs["escalate"] = df_pairs["rbs_conf"] < TAU
log(f"τ={TAU}, escalations={int(df_pairs['escalate'].sum()):,} / {len(df_pairs):,} ({df_pairs['escalate'].mean()*100:.1f}%)")


[16:41:45] τ=0.7, escalations=8,035 / 8,035 (100.0%)


## 12) Lexical Agent
**Tujuan:** Mendeteksi cue restriction (only/once/never/...) dan asimetri.


In [16]:
# =================
# 12) Lexical Agent
# =================
LEX_CUES = ["only","once","at most","never","must","shall","cannot"]

def lexical_agent(text1: str, text2: str) -> Dict[str,Any]:
    low1, low2 = text1.lower(), text2.lower()
    hits1 = [c for c in LEX_CUES if c in low1]
    hits2 = [c for c in LEX_CUES if c in low2]
    asym = (len(hits1) > 0) != (len(hits2) > 0)
    if asym:
        return {"lex_label":"Conflict","lex_conf":0.85,"lex_evidence":{"hits_1":hits1,"hits_2":hits2,"asym":True}}
    return {"lex_label":"Non-Conflict","lex_conf":0.60,"lex_evidence":{"hits_1":hits1,"hits_2":hits2,"asym":False}}

lex = df_pairs.apply(lambda r: pd.Series(lexical_agent(r["text_1"], r["text_2"])), axis=1)
df_pairs = pd.concat([df_pairs, lex], axis=1)
display(df_pairs[["req_id_1","req_id_2","lex_label","lex_conf","lex_evidence"]].head(5))


,req_id_1,req_id_2,lex_label,lex_conf,lex_evidence
0,44,79,Non-Conflict,0.6,"{'hits_1': ['shall'], 'hits_2': ['shall'], 'as..."
1,44,80,Non-Conflict,0.6,"{'hits_1': ['shall'], 'hits_2': ['shall'], 'as..."
2,79,80,Non-Conflict,0.6,"{'hits_1': ['shall'], 'hits_2': ['shall'], 'as..."
3,29,30,Non-Conflict,0.6,"{'hits_1': ['shall'], 'hits_2': ['shall'], 'as..."
4,29,31,Non-Conflict,0.6,"{'hits_1': ['shall'], 'hits_2': ['shall'], 'as..."


## 13) Contextual Agent
**Tujuan:** Menilai konteks berdasar overlap tuple & similarity teks.


In [17]:
# ====================
# 13) Contextual Agent
# ====================
def tuple_overlap(t1: Dict[str,str], t2: Dict[str,str]) -> float:
    keys = ["agent","operation","input","output","restriction","event"]
    a = [_norm(t1.get(k,"")) for k in keys]
    b = [_norm(t2.get(k,"")) for k in keys]
    same = sum(1 for i in range(len(keys)) if a[i] and b[i] and a[i]==b[i])
    denom = sum(1 for i in range(len(keys)) if a[i] or b[i])
    return same / max(1, denom)

def contextual_agent(t1: Dict[str,str], t2: Dict[str,str], text1: str, text2: str) -> Dict[str,Any]:
    s1, s2 = set(_norm(text1).split()), set(_norm(text2).split())
    j = len(s1 & s2) / max(1, len(s1 | s2))
    to = tuple_overlap(t1, t2)
    score = 0.5*j + 0.5*to
    if score >= 0.35:
        return {"ctx_label":"Conflict","ctx_conf":float(min(0.90, 0.60+score)),"ctx_evidence":{"jaccard":j,"tuple_overlap":to,"score":score}}
    return {"ctx_label":"Non-Conflict","ctx_conf":0.55,"ctx_evidence":{"jaccard":j,"tuple_overlap":to,"score":score}}

ctx = df_pairs.apply(lambda r: pd.Series(contextual_agent(r["tuple_1"], r["tuple_2"], r["text_1"], r["text_2"])), axis=1)
df_pairs = pd.concat([df_pairs, ctx], axis=1)
display(df_pairs[["req_id_1","req_id_2","ctx_label","ctx_conf","ctx_evidence"]].head(5))


,req_id_1,req_id_2,ctx_label,ctx_conf,ctx_evidence
0,44,79,Conflict,0.9,"{'jaccard': 0.5555555555555556, 'tuple_overlap..."
1,44,80,Conflict,0.9,"{'jaccard': 0.45454545454545453, 'tuple_overla..."
2,79,80,Conflict,0.9,"{'jaccard': 0.5862068965517241, 'tuple_overlap..."
3,29,30,Conflict,0.9,"{'jaccard': 1.0, 'tuple_overlap': 1.0, 'score'..."
4,29,31,Conflict,0.9,"{'jaccard': 0.9, 'tuple_overlap': 0.75, 'score..."


## 14) Logic Agent
**Tujuan:** Audit ulang berbasis RBS + ConfRBS.


In [18]:
# ==============
# 14) Logic Agent
# ==============
def logic_agent(active_rules: List[str], rbs_conf: float) -> Dict[str,Any]:
    if not active_rules:
        return {"logic_label":"Non-Conflict","logic_conf":0.60,"logic_trace":{"active_rules":[]}}
    label = "Conflict" if rbs_conf >= 0.50 else "Non-Conflict"
    return {"logic_label":label,"logic_conf":float(min(0.95, max(0.55, rbs_conf))),"logic_trace":{"active_rules":active_rules,"conf_rbs":rbs_conf}}

lg = df_pairs.apply(lambda r: pd.Series(logic_agent(r["active_rules"], float(r["rbs_conf"]))), axis=1)
df_pairs = pd.concat([df_pairs, lg], axis=1)
display(df_pairs[["req_id_1","req_id_2","logic_label","logic_conf","logic_trace"]].head(5))


,req_id_1,req_id_2,logic_label,logic_conf,logic_trace
0,44,79,Non-Conflict,0.60,{'active_rules': []}
1,44,80,Non-Conflict,0.60,{'active_rules': []}
2,79,80,Non-Conflict,0.55,"{'active_rules': ['R1', 'R3', 'R6'], 'conf_rbs..."
3,29,30,Non-Conflict,0.55,"{'active_rules': ['R1', 'R3', 'R6'], 'conf_rbs..."
4,29,31,Non-Conflict,0.60,{'active_rules': []}


## 15C–15D) Consensus Agent (Precision-based weighted voting)
**Tujuan:** Hitung bobot agent dari precision (TP/(TP+FP)) di data validasi, lalu voting final.


In [19]:
# =====================================
# 15C) Agent weights based on precision
# =====================================
def agent_precision(df: pd.DataFrame, label_col: str) -> float:
    active = df[label_col].astype(str).str.lower().eq("conflict")
    tp = int(((active == True) & (df["y_true_pair"] == 1)).sum())
    fp = int(((active == True) & (df["y_true_pair"] == 0)).sum())
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

# compute on validation split (align columns)
df_val = df_pairs.loc[df_val.index].copy()
w_agent = {
    "lex": agent_precision(df_val, "lex_label"),
    "ctx": agent_precision(df_val, "ctx_label"),
    "logic": agent_precision(df_val, "logic_label"),
}
df_w_agent = pd.DataFrame([{"agent":k,"w_agent":v} for k,v in w_agent.items()]).sort_values("agent")
save_df(df_w_agent, "agent_weights_precision.csv")
display(df_w_agent)

# ==================================
# 15D) Weighted voting final decision
# ==================================
def consensus(row: pd.Series) -> Dict[str,Any]:
    # accept RBS if not escalated
    if not bool(row["escalate"]):
        return {"final_label": row["rbs_label"], "final_conf": float(row["rbs_conf"]),
                "final_rationale": {"accepted_rbs": True, "tau": TAU}}

    score_c = 0.0
    score_n = 0.0

    # lex
    if str(row["lex_label"]).lower() == "conflict":
        score_c += w_agent["lex"] * float(row["lex_conf"])
    else:
        score_n += w_agent["lex"] * float(row["lex_conf"])

    # ctx
    if str(row["ctx_label"]).lower() == "conflict":
        score_c += w_agent["ctx"] * float(row["ctx_conf"])
    else:
        score_n += w_agent["ctx"] * float(row["ctx_conf"])

    # logic
    if str(row["logic_label"]).lower() == "conflict":
        score_c += w_agent["logic"] * float(row["logic_conf"])
    else:
        score_n += w_agent["logic"] * float(row["logic_conf"])

    if score_c >= score_n:
        final_label = "Conflict"
        final_conf = score_c / max(1e-9, (score_c + score_n))
    else:
        final_label = "Non-Conflict"
        final_conf = score_n / max(1e-9, (score_c + score_n))

    rationale = {
        "accepted_rbs": False,
        "tau": TAU,
        "score_conflict": score_c,
        "score_nonconflict": score_n,
        "w_agent": w_agent,
        "agent_votes": {
            "lex": {"label": row["lex_label"], "conf": float(row["lex_conf"])},
            "ctx": {"label": row["ctx_label"], "conf": float(row["ctx_conf"])},
            "logic": {"label": row["logic_label"], "conf": float(row["logic_conf"])},
        }
    }
    return {"final_label": final_label, "final_conf": float(final_conf), "final_rationale": rationale}

cs = df_pairs.apply(lambda r: pd.Series(consensus(r)), axis=1)
df_pairs = pd.concat([df_pairs, cs], axis=1)

display(df_pairs[["req_id_1","req_id_2","escalate","rbs_label","rbs_conf","final_label","final_conf"]].head(10))


[16:41:53] Saved: outputs/agent_weights_precision.csv (3 rows)


,agent,w_agent
1,ctx,0.017843
0,lex,0.000000
2,logic,0.000000


,req_id_1,req_id_2,escalate,rbs_label,rbs_conf,final_label,final_conf
0,44,79,True,Non-Conflict,0.000000,Conflict,1.0
1,44,80,True,Non-Conflict,0.000000,Conflict,1.0
2,79,80,True,Conflict,0.043291,Conflict,1.0
3,29,30,True,Conflict,0.043291,Conflict,1.0
4,29,31,True,Non-Conflict,0.000000,Conflict,1.0
5,29,32,True,Non-Conflict,0.000000,Conflict,1.0
6,29,87,True,Non-Conflict,0.000000,Conflict,1.0
7,29,88,True,Non-Conflict,0.000000,Conflict,1.0
8,29,89,True,Conflict,0.043291,Conflict,1.0
9,29,90,True,Conflict,0.043291,Conflict,1.0


## 16) Explainability Agent
**Tujuan:** Menghasilkan penjelasan 2–5 kalimat: evidence lexical, tuple, active rules + bobot, dan rationale voting.


In [20]:
# ======================
# 16) Explainability Agent
# ======================
def explainability(row: pd.Series) -> str:
    t1, t2 = row["tuple_1"], row["tuple_2"]
    ar = row["active_rules"]
    wr = {r: float(w_rule.get(r,0.0)) for r in ar}
    sents = []

    sents.append(f"Pair ({row['req_id_1']}, {row['req_id_2']}) classified as {row['final_label']} (confidence={row['final_conf']:.2f}).")
    if ar:
        sents.append(f"RBS activated {ar} with weights {wr} and Conf_RBS={float(row['rbs_conf']):.2f}.")
    else:
        sents.append(f"RBS found no active rules (Conf_RBS={float(row['rbs_conf']):.2f}).")

    ev = row.get("lex_evidence", {})
    if isinstance(ev, dict) and (ev.get("hits_1") or ev.get("hits_2")):
        sents.append(f"Lexical cues: {ev.get('hits_1', [])} vs {ev.get('hits_2', [])} (asym={ev.get('asym')}).")

    sents.append(
        "Tuple summary: "
        f"agent='{t1.get('agent','')}' op='{t1.get('operation','')}' in='{t1.get('input','')}' out='{t1.get('output','')}' res='{t1.get('restriction','')}' "
        "vs "
        f"agent='{t2.get('agent','')}' op='{t2.get('operation','')}' in='{t2.get('input','')}' out='{t2.get('output','')}' res='{t2.get('restriction','')}'."
    )

    rat = row.get("final_rationale", {})
    if isinstance(rat, dict) and not rat.get("accepted_rbs", True):
        sents.append(f"Escalated (τ={rat.get('tau')}); vote scores conflict={rat.get('score_conflict'):.3f} vs nonconflict={rat.get('score_nonconflict'):.3f}.")
    else:
        sents.append(f"Not escalated (τ={TAU}); accepted RBS decision.")

    return " ".join(sents[:5])

df_pairs["explainability_text"] = df_pairs.apply(explainability, axis=1)
display(df_pairs[["req_id_1","req_id_2","final_label","final_conf","explainability_text"]].head(3))


,req_id_1,req_id_2,final_label,final_conf,explainability_text
0,44,79,Conflict,1.0,"Pair (44, 79) classified as Conflict (confiden..."
1,44,80,Conflict,1.0,"Pair (44, 80) classified as Conflict (confiden..."
2,79,80,Conflict,1.0,"Pair (79, 80) classified as Conflict (confiden..."


## 17) Final Output
Ekspor:
- File 1: **Conflict Pair, Confidence Score, Explainability**
- File 2: audit rule + vote


In [21]:
# =========================
# 17) Final Output (Exports)
# =========================
df_results = df_pairs.copy()

df_out_main = pd.DataFrame({
    "Conflict Pair": df_results.apply(lambda r: f"({r['req_id_1']}, {r['req_id_2']})", axis=1),
    "Confidence Score": df_results["final_conf"].astype(float),
    "Explainability": df_results["explainability_text"].astype(str),
})
save_df(df_out_main, "FINAL_output_main.csv")

df_out_audit = df_results[[
    "req_id_1","req_id_2","method","embedding_type","cluster_id",
    "active_rules","rbs_label","rbs_rule","rbs_conf","rbs_explain","escalate",
    "lex_label","lex_conf","lex_evidence",
    "ctx_label","ctx_conf","ctx_evidence",
    "logic_label","logic_conf","logic_trace",
    "final_label","final_conf","final_rationale"
]].copy()
df_out_audit["w_rule_snapshot"] = df_out_audit["active_rules"].apply(lambda ar: {r: float(w_rule.get(r,0.0)) for r in ar})
df_out_audit["w_agent_snapshot"] = json.dumps(w_agent)

save_df(df_out_audit, "FINAL_output_audit.csv")

display(df_out_main.head())


[16:41:57] Saved: outputs/FINAL_output_main.csv (8,035 rows)
[16:41:57] Saved: outputs/FINAL_output_audit.csv (8,035 rows)


,Conflict Pair,Confidence Score,Explainability
0,"(44, 79)",1.0,"Pair (44, 79) classified as Conflict (confiden..."
1,"(44, 80)",1.0,"Pair (44, 80) classified as Conflict (confiden..."
2,"(79, 80)",1.0,"Pair (79, 80) classified as Conflict (confiden..."
3,"(29, 30)",1.0,"Pair (29, 30) classified as Conflict (confiden..."
4,"(29, 31)",1.0,"Pair (29, 31) classified as Conflict (confiden..."


## 18) Evaluation
**Tujuan:** Bandingkan baseline (Clustering+RBS) vs proposed (Clustering+RBS+Multi-Agent).

**Catatan:** Evaluasi pair-level terbaik jika kamu punya label pasangan ground-truth; di sini memakai proxy.


In [22]:
# =========================
# 18) Evaluation
# =========================
def eval_predictions(df: pd.DataFrame, y_true_col: str, label_col: str) -> Dict[str,float]:
    y_true = df[y_true_col].astype(int)
    y_pred = df[label_col].astype(str).str.lower().map({"conflict":1,"non-conflict":0,"nonconflict":0,"neutral":0}).fillna(0).astype(int)
    p,r,f1,_ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {"precision": float(p), "recall": float(r), "f1": float(f1)}

baseline = eval_predictions(df_test.assign(rbs_label=df_pairs.loc[df_test.index,"rbs_label"]), "y_true_pair", "rbs_label")
proposed = eval_predictions(df_test.assign(final_label=df_pairs.loc[df_test.index,"final_label"]), "y_true_pair", "final_label")

df_eval = pd.DataFrame([
    {"system":"baseline_clustering_rbs", **baseline},
    {"system":"proposed_clustering_rbs_multiagent", **proposed},
])
save_df(df_eval, "evaluation_baseline_vs_proposed.csv")
display(df_eval)

total_pairs_all = len(df_req) * (len(df_req)-1) // 2
pair_reduction = 1 - (len(df_pairs) / max(1, total_pairs_all))
log(f"Total possible all-vs-all pairs: {total_pairs_all:,}")
log(f"Intra-cluster pairs produced: {len(df_pairs):,}")
log(f"Pair reduction: {pair_reduction*100:.2f}%")


[16:41:57] Saved: outputs/evaluation_baseline_vs_proposed.csv (2 rows)


,system,precision,recall,f1
0,baseline_clustering_rbs,0.026568,0.223602,0.047493
1,proposed_clustering_rbs_multiagent,0.023964,0.434783,0.045425


[16:41:57] Total possible all-vs-all pairs: 6,786
[16:41:57] Intra-cluster pairs produced: 8,035
[16:41:57] Pair reduction: -18.41%


# 🔥 MASTER PROMPT (Reference)
Gabungkan semua tahap (1–18) menjadi 1 notebook Google Colab lengkap dengan:

- Dual embedding
- Grid search clustering
- Sarwosri labeling
- RBS identik tabel
- Conf_RBS probabilistik
- Threshold τ=0.7
- Multi-agent escalation
- Precision-based weighted consensus
- Final 3 kolom output
- Audit file
- Modular functions
- Markdown heading tiap tahap
